In [1]:
import pandas as pd
import numpy as np

In [2]:
# 读取CSV文件
# /Users/zhanyu/code/data-hh/海航系销售结果数据
df = pd.read_csv('../../data-hh/海航系销售结果数据/mid_flt_allmkt_segment_result_sale_tmp_202410221701.csv')

# 显示前两行数据以确保正确加载
print(df.shape)
print(df.head(5))

(733570, 15)
     flt_date                    segment        flt_no bd_type  dep_time  cap  \
0  2024-05-01  tKjndGSl9NQ=-h9GisD/ZayE=  Mg9RILDB+wE=      窄体  07:05:00    0   
1  2024-05-01  tKjndGSl9NQ=-Lue5PP9SfQU=  Mg9RILDB+wE=      窄体  07:05:00  164   
2  2024-05-01  h9GisD/ZayE=-tKjndGSl9NQ=  Tc/K+Vr3H50=      窄体  13:00:00    0   
3  2024-05-01  Lue5PP9SfQU=-tKjndGSl9NQ=  Tc/K+Vr3H50=      窄体  15:30:00  164   
4  2024-05-01  h9GisD/ZayE=-Lue5PP9SfQU=  Tc/K+Vr3H50=      窄体  13:00:00  164   

  aircraft  legs  leg_no  duration  tkt_rev  pax             a             b  \
0      320     3       3      0.00  40900.0   41  tKjndGSl9NQ=  Lue5PP9SfQU=   
1      320     3       1      2.08  79380.0  121  tKjndGSl9NQ=  Lue5PP9SfQU=   
2      320     3       3      0.00  61915.0   52  h9GisD/ZayE=  Lue5PP9SfQU=   
3      320     3       2      1.90  50248.0  101  h9GisD/ZayE=  Lue5PP9SfQU=   
4      320     3       1      1.27  56389.0  107  h9GisD/ZayE=  Lue5PP9SfQU=   

              c  
0

In [3]:
# 只保留2024年6月的数据
df = df[df['flt_date'].str.startswith('2024-06')]

# 查看筛选后的数据
print("筛选后数据形状：", df.shape)
print("\n前5行数据：")
print(df.head())

筛选后数据形状： (364259, 15)

前5行数据：
          flt_date                    segment        flt_no bd_type  dep_time  \
369311  2024-06-01  g1hKMyP1V8I=-LIrnUnlANso=  khm3bimM5JM=      窄体  17:20:00   
369312  2024-06-01  LIrnUnlANso=-g1hKMyP1V8I=  sE525ybKZIw=      窄体  20:20:00   
369313  2024-06-01  g1hKMyP1V8I=-KfwFeosCXek=  b28ihE1qgZc=      窄体  10:40:00   
369314  2024-06-01  KfwFeosCXek=-g1hKMyP1V8I=  Zdzst69JhpI=      窄体  15:55:00   
369315  2024-06-01  EIsXRbFUH34=-f1DuEztFfn8=  hxp4q4QvRCM=      支线  12:30:00   

        cap aircraft  legs  leg_no  duration  tkt_rev  pax             a  \
369311  170      738     1       1      2.12  68018.0  145  g1hKMyP1V8I=   
369312  170      738     1       1      1.83  46231.0  121  LIrnUnlANso=   
369313    0      738     3       3      0.00  25997.0   31  g1hKMyP1V8I=   
369314    0      738     3       3      0.00   1415.0    5  KfwFeosCXek=   
369315   89      CRJ     3       1      0.60  13956.0   27  EIsXRbFUH34=   

                   b      

In [4]:
# 保存为新的CSV文件
df.to_csv('../../data-hh/my/hh_result/hh_result_2024_6_原始.csv', index=False)

print("筛选后数据形状：", df.shape)
print("文件已保存至: ../../data-hh/my/hh_result/hh_result_2024_6_原始.csv")

筛选后数据形状： (364259, 15)
文件已保存至: ../../data-hh/my/hh_result/hh_result_2024_6_原始.csv


In [5]:
# 只保留legs=3的行
df = df[df['legs'] == 3]

In [6]:
# 统计legs的值计数
legs_count = df['legs'].value_counts().sort_index()

# 显示统计结果
print("Legs分布情况：")
print(legs_count)

# 显示百分比
legs_percentage = df['legs'].value_counts(normalize=True).sort_index() * 100
print("\nLegs分布百分比：")
print(legs_percentage.round(2), "%")

Legs分布情况：
legs
3    136257
Name: count, dtype: int64

Legs分布百分比：
legs
3    100.0
Name: proportion, dtype: float64 %


In [7]:
# 定义需要添加后缀的列
suffix_columns = ['dep_time', 'cap', 'duration', 'tkt_rev', 'pax']
# 定义不需要改变的列
keep_columns = ['flt_date', 'flt_no', 'bd_type', 'aircraft', 'a', 'b', 'c']

# 创建一个空的DataFrame来存储结果
result = []

# 按照flt_date和flt_no分组
for (flt_date, flt_no), group in df.groupby(['flt_date', 'flt_no']):
    # 初始化新行数据
    new_row = {}
    
    # 处理不变的列
    for col in keep_columns:
        new_row[col] = group[col].iloc[0]
    
    # 处理需要添加后缀的列
    for col in suffix_columns:
        # 按leg_no排序确保顺序正确
        sorted_group = group.sort_values('leg_no')
        for _, row in sorted_group.iterrows():
            leg_no = row['leg_no']
            new_row[f'{col}_{leg_no}'] = row[col]
    
    result.append(new_row)

# 转换结果为DataFrame
result_df = pd.DataFrame(result)

# 保存结果
result_df.to_csv('../../data-hh/my/hh_result/hh_result_2024_6_merged.csv', index=False)

合并后数据形状： (41410, 22)

前5行数据：
     flt_date        flt_no bd_type aircraft             a             b  \
0  2024-06-01  +/Z/tRXgCf4=      支线      ARJ  F6QbezoeygU=  MvSmXAgLriA=   
1  2024-06-01  +/eNZWocBU0=      窄体      320  nE/SoZJHO24=  g1hKMyP1V8I=   
2  2024-06-01  +2LS+LVYdZk=      窄体      738  DUbxF00Y8a0=  N4hmDZN/CJQ=   
3  2024-06-01  +2OUR9mFJuw=      窄体      321  Lue5PP9SfQU=  wSF4qXvI044=   
4  2024-06-01  +9XajuFENVg=      窄体      738  Szk6NJM5C9Y=  FWD6O2fHsIU=   

              c dep_time_1 dep_time_2 dep_time_3  ...  cap_3  duration_1  \
0  dpTE9RO47aU=   08:15:00   10:10:00   08:15:00  ...    0.0        0.85   
1  CnmNtb5MOJA=   16:45:00   21:35:00   16:45:00  ...    0.0        2.47   
2  QyEgIt0M8Sg=   14:15:00   17:15:00   14:15:00  ...    0.0        1.98   
3  xIqe+QQIQXQ=        NaN   22:45:00   18:40:00  ...    0.0         NaN   
4  XUXDuFGVfBM=   17:25:00   20:20:00   17:25:00  ...    0.0        1.33   

   duration_2  duration_3  tkt_rev_1  tkt_rev_2  tkt_rev_

In [8]:
# 查看结果
# 设置pandas显示选项，显示所有列
pd.set_option('display.max_columns', None)  # 显示所有列
print("合并后数据形状：", result_df.shape)
print("\n前5行数据：")
print(result_df.head())

合并后数据形状： (41410, 22)

前5行数据：
     flt_date        flt_no bd_type aircraft             a             b  \
0  2024-06-01  +/Z/tRXgCf4=      支线      ARJ  F6QbezoeygU=  MvSmXAgLriA=   
1  2024-06-01  +/eNZWocBU0=      窄体      320  nE/SoZJHO24=  g1hKMyP1V8I=   
2  2024-06-01  +2LS+LVYdZk=      窄体      738  DUbxF00Y8a0=  N4hmDZN/CJQ=   
3  2024-06-01  +2OUR9mFJuw=      窄体      321  Lue5PP9SfQU=  wSF4qXvI044=   
4  2024-06-01  +9XajuFENVg=      窄体      738  Szk6NJM5C9Y=  FWD6O2fHsIU=   

              c dep_time_1 dep_time_2 dep_time_3  cap_1  cap_2  cap_3  \
0  dpTE9RO47aU=   08:15:00   10:10:00   08:15:00   78.0   78.0    0.0   
1  CnmNtb5MOJA=   16:45:00   21:35:00   16:45:00  158.0  158.0    0.0   
2  QyEgIt0M8Sg=   14:15:00   17:15:00   14:15:00  170.0  170.0    0.0   
3  xIqe+QQIQXQ=        NaN   22:45:00   18:40:00    NaN  195.0    0.0   
4  XUXDuFGVfBM=   17:25:00   20:20:00   17:25:00  178.0  178.0    0.0   

   duration_1  duration_2  duration_3  tkt_rev_1  tkt_rev_2  tkt_rev_3  pax

In [9]:
# 保存结果
result_df.to_csv('../../data-hh/my/hh_result/hh_result_2024_6_merged.csv', index=False)